# Razorpay RiskOS — RiskAuditor-7B Model Training Notebook
### Fine-Tuning & Verifiable Reinforcement Learning (GRPO) for Track 2: AI Risk Manager

**Hardware**: 1x T4 GPU (Google Colab Free)
**Stack**: HuggingFace `transformers` + `peft` (QLoRA) + `trl` + `bitsandbytes`
**Training Time**: ~25 minutes

In [ ]:
# 1. Clean, Wheel-Free Installation (No compilation or xformers errors)
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate

In [ ]:
# 2. Download Training Data directly from GitHub (if not already uploaded)
import os
if not os.path.exists("train.jsonl"):
    !curl -s -L -o train.jsonl "https://raw.githubusercontent.com/Kanhaiya76618/Research-OS/main/backend/lib/train/data/train.jsonl"
    print("[✓] Downloaded train.jsonl directly from GitHub!")
else:
    print("[✓] train.jsonl already present.")

In [ ]:
# 3. Load Qwen-2.5-7B with 4-bit Quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

# 4. Attach LoRA Adapter Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Format Dataset & Run Training with TRL SFTTrainer
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

raw_dataset = load_dataset("json", data_files="train.jsonl")["train"]

def format_prompts(batch):
    texts = []
    for p, c in zip(batch["prompt"], batch["completion"]):
        texts.append(f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n{c}<|im_end|>")
    return {"text": texts}

train_dataset = raw_dataset.map(format_prompts, batched=True)

training_args = TrainingArguments(
    output_dir="./riskauditor_outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim="paged_adamw_8bit",
    save_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=1536,
    args=training_args,
)

print("Starting GPU Training on Qwen-2.5-7B LoRA...")
trainer.train()

In [ ]:
# 6. Save Trained LoRA Adapter Weights
model.save_pretrained("riskauditor_7b_lora")
tokenizer.save_pretrained("riskauditor_7b_lora")
print("[✓] Training Complete! LoRA Adapter weights saved to riskauditor_7b_lora/")